# 🧠 Satria Data 2025 — NLP Emotion Classification

**Dataset:** MELD (Multimodal EmotionLines Dataset)  
**Task:** Emotion Recognition dari teks percakapan  
**Emosi:** Anger, Disgust, Fear, Joy, Neutral, Sadness, Surprise (7 kelas)  

---

## Section 1: Setup Environment

In [ ]:
# === Import Libraries ===
# Kaggle sudah punya semua library ini (termasuk wordcloud), tidak perlu pip install

import os
import re
import string
import warnings
import random
from collections import Counter

import numpy as np
import pandas as pd

# ML (scikit-learn) — hanya import yang benar-benar dipakai
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score
)

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Visualisasi — matplotlib + seaborn saja (lebih ringan dari plotly)
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('✅ Semua library berhasil diimport!')

In [ ]:
# === Setup NLTK Data (Kaggle-compatible, no internet needed) ===

# Kaggle sudah punya NLTK data pre-installed di path ini
kaggle_nltk_paths = ['/usr/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/share/nltk_data']
for p in kaggle_nltk_paths:
    if os.path.isdir(p) and p not in nltk.data.path:
        nltk.data.path.insert(0, p)

# Cek apakah data sudah ada, download hanya kalau belum
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    try:
        nltk.data.find(f'tokenizers/{pkg}' if 'punkt' in pkg else f'corpora/{pkg}')
        print(f'  OK: {pkg} sudah ada')
    except LookupError:
        try:
            downloaded = nltk.download(pkg, quiet=True)
            if downloaded:
                print(f'  OK: {pkg} berhasil download')
            else:
                print(f'  SKIP: {pkg} gagal download (Kaggle offline)')
        except Exception as e:
            print(f'  SKIP: {pkg} tidak tersedia ({e})')

# Fallback: hardcoded English stopwords kalau NLTK stopwords gagal
try:
    from nltk.corpus import stopwords as _sw
    STOP_WORDS = set(_sw.words('english'))
    print(f'\nNLTK stopwords loaded ({len(STOP_WORDS)} words)')
except Exception:
    STOP_WORDS = {
        'i','me','my','myself','we','our','ours','ourselves','you',"you're","you've",
        "you'll","you'd",'your','yours','yourself','yourselves','he','him','his',
        'himself','she',"she's",'her','hers','herself','it',"it's",'its','itself',
        'they','them','their','theirs','themselves','what','which','who','whom',
        'this','that',"that'll",'these','those','am','is','are','was','were','be',
        'been','being','have','has','had','having','do','does','did','doing','a',
        'an','the','and','but','if','or','because','as','until','while','of','at',
        'by','for','with','about','against','between','through','during','before',
        'after','above','below','to','from','up','down','in','out','on','off',
        'over','under','again','further','then','once','here','there','when',
        'where','why','how','all','both','each','few','more','most','other','some',
        'such','no','nor','not','only','own','same','so','than','too','very','s',
        't','can','will','just','don',"don't",'should',"should've",'now','d','ll',
        'm','o','re','ve','y','ain','aren',"aren't",'couldn',"couldn't",'didn',
        "didn't",'doesn',"doesn't",'hadn',"hadn't",'hasn',"hasn't",'haven',
        "haven't",'isn',"isn't",'ma','mightn',"mightn't",'mustn',"mustn't",
        'needn',"needn't",'shan',"shan't",'shouldn',"shouldn't",'wasn',"wasn't",
        'weren',"weren't",'won',"won't",'wouldn',"wouldn't"
    }
    print(f'\nNLTK stopwords tidak tersedia, pakai hardcoded ({len(STOP_WORDS)} words)')

print('\nNLTK setup selesai!')


In [ ]:
# === Konfigurasi Global ===

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Matplotlib & Seaborn style
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12
})

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Label emosi MELD
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
EMOTION_COLORS = {
    'anger': '#E74C3C', 'disgust': '#8E44AD', 'fear': '#2C3E50',
    'joy': '#F1C40F', 'neutral': '#95A5A6', 'sadness': '#3498DB',
    'surprise': '#E67E22'
}

print(f'Device : {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print(f'Seed   : {RANDOM_SEED}')
print(f'PyTorch: {torch.__version__}')
print('✅ Konfigurasi selesai!')

---

## Section 2: Data Loading

Dataset MELD sudah ditambahkan sebagai Kaggle Input (`zaber666/meld-dataset`).

In [ ]:
# === Load MELD Dataset ===
# Path langsung ke CSV — lebih cepat daripada os.walk

MELD_RAW = '/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw'

# Kolom yang dibutuhkan untuk task NLP (abaikan kolom video/audio)
USE_COLS = ['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment',
            'Dialogue_ID', 'Utterance_ID', 'Season', 'Episode']

# Definisi path langsung
csv_paths = {
    'train': os.path.join(MELD_RAW, 'train', 'train_sent_emo.csv'),
    'dev':   os.path.join(MELD_RAW, 'dev_sent_emo.csv'),
    'test':  os.path.join(MELD_RAW, 'test_sent_emo.csv'),
}

# Fallback: kalau train CSV ada di level atas
if not os.path.isfile(csv_paths['train']):
    alt = os.path.join(MELD_RAW, 'train_sent_emo.csv')
    if os.path.isfile(alt):
        csv_paths['train'] = alt

# Load dengan usecols untuk hemat memory
datasets = {}
for split, path in csv_paths.items():
    if os.path.isfile(path):
        # Cek kolom yang tersedia dulu
        available_cols = pd.read_csv(path, nrows=0).columns.tolist()
        cols_to_use = [c for c in USE_COLS if c in available_cols]
        
        datasets[split] = pd.read_csv(path, usecols=cols_to_use)
        print(f'✅ {split:5s} loaded: {datasets[split].shape}  <- {os.path.basename(path)}')
    else:
        print(f'⚠️ {split:5s} NOT FOUND: {path}')

df_train = datasets.get('train')
df_dev   = datasets.get('dev')
df_test  = datasets.get('test')

if df_train is not None:
    print(f'\n--- Train Preview ---')
    display(df_train.head())
    print(f'\nKolom: {df_train.columns.tolist()}')
    print(f'Missing values:\n{df_train.isnull().sum()[df_train.isnull().sum() > 0]}')

In [ ]:
# === Quick Data Summary ===

for name, df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
    if df is not None:
        print(f'\n📊 {name}: {len(df)} utterances, '
              f'{df["Dialogue_ID"].nunique() if "Dialogue_ID" in df.columns else "?"} dialogues')
        if 'Emotion' in df.columns:
            print(df['Emotion'].value_counts().to_string())

In [ ]:
# === Optimize dtypes untuk hemat memory ===

def optimize_df(df):
    """Downcast numeric kolom & convert string columns ke category."""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in ['Emotion', 'Sentiment', 'Speaker']:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df

for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
    if df is not None:
        mem_before = df.memory_usage(deep=True).sum() / 1024**2
        optimize_df(df)
        mem_after = df.memory_usage(deep=True).sum() / 1024**2
        print(f'{name}: {mem_before:.2f} MB -> {mem_after:.2f} MB '
              f'({(1 - mem_after/mem_before)*100:.0f}% reduced)')

## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# === 3.1 Distribusi Emosi ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]):
    if df is not None and 'Emotion' in df.columns:
        counts = df['Emotion'].value_counts()
        colors = [EMOTION_COLORS.get(e, '#999') for e in counts.index]
        ax.barh(counts.index, counts.values, color=colors)
        ax.set_title(f'{name} ({len(df)} samples)')
        ax.set_xlabel('Count')
        for i, v in enumerate(counts.values):
            ax.text(v + 10, i, f'{v} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)
plt.suptitle('Distribusi Emosi per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 3.2 Panjang Utterance per Emosi ===
df_train['text_len'] = df_train['Utterance'].astype(str).str.len()
df_train['word_count'] = df_train['Utterance'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=df_train, x='Emotion', y='text_len', ax=axes[0],
            palette=EMOTION_COLORS, order=EMOTION_LABELS)
axes[0].set_title('Panjang Karakter per Emosi')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df_train, x='Emotion', y='word_count', ax=axes[1],
            palette=EMOTION_COLORS, order=EMOTION_LABELS)
axes[1].set_title('Jumlah Kata per Emosi')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f'Rata-rata panjang: {df_train["word_count"].mean():.1f} kata')
print(f'Max: {df_train["word_count"].max()} kata')

In [ ]:
# === 3.3 Word Cloud per Emosi ===
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for i, emotion in enumerate(EMOTION_LABELS):
    text = ' '.join(df_train[df_train['Emotion'] == emotion]['Utterance'].astype(str))
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap='Set2', max_words=80).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(emotion.capitalize(), fontsize=13, fontweight='bold',
                      color=EMOTION_COLORS[emotion])
    axes[i].axis('off')
axes[-1].axis('off')
plt.suptitle('Word Cloud per Emosi', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 3.4 Top Speakers ===
if 'Speaker' in df_train.columns:
    top_speakers = df_train['Speaker'].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 4))
    top_speakers.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 10 Speakers (Train)')
    ax.set_xlabel('Jumlah Utterance')
    plt.tight_layout()
    plt.show()

## Section 4: Text Preprocessing & Feature Engineering

In [ ]:
# === 4.1 Text Cleaning ===
from nltk.stem import WordNetLemmatizer

# Define negation words to exclude from stopwords
negation_words = {
    'no', 'not', 'nor', 'neither', 'never', 'none', 'but', 'against',
    "don't", "can't", "won't", "shouldn't", "couldn't", "didn't",
    "doesn't", "haven't", "hasn't", "hadn't", "isn't", "aren't", "wasn't", "weren't"
}
STOP_WORDS = set(stopwords.words('english')) - negation_words

CONTRACTIONS_MAP = {
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are", "i've": "i have",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "can't": "cannot", "won't": "will not", "wouldn't": "would not",
    "shouldn't": "should not", "couldn't": "could not"
}

lemmatizer = WordNetLemmatizer()
try:
    nltk.data.find('corpora/wordnet')
    USE_LEMMATIZER = True
except LookupError:
    USE_LEMMATIZER = False
    print('⚠️ WordNet tidak ditemukan (offline mode). Lemmatization akan di-skip.')

def clean_text(text):
    """Bersihkan teks: lowercase, expand contractions, hapus punctuation (keep !?), lemmatize & filter stopwords."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    # Expand contractions
    for contraction, expansion in CONTRACTIONS_MAP.items():
        text = text.replace(contraction, expansion)
    text = re.sub(r'http\S+|www\S+', '', text)       # hapus URL
    text = re.sub(r'[^a-z\s!?]', ' ', text)            # hanya huruf & !?
    text = re.sub(r'([!?])', r' \1 ', text)           # pad !? dengan spasi
    text = re.sub(r'\s+', ' ', text).strip()           # hapus extra spaces
    tokens = text.split()
    # Lemmatize and filter stopwords
    if USE_LEMMATIZER:
        tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 1 or t in ['!', '?']]
    else:
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1 or t in ['!', '?']]
    return ' '.join(tokens)

# Apply ke semua split
for df in [df_train, df_dev, df_test]:
    if df is not None:
        df['clean_text'] = df['Utterance'].apply(clean_text)

print(f'Contoh cleaning:')
for i in range(3):
    print(f'  Original : {df_train.iloc[i]["Utterance"]}')
    print(f'  Clean    : {df_train.iloc[i]["clean_text"]}\n')

In [ ]:
# === 4.2 Hapus empty rows & Tambah Dialogue Context ===

# Urutkan berdasarkan Dialogue_ID & Utterance_ID untuk memastikan runtutan dialog benar
if df_train is not None:
    df_train = df_train.sort_values(['Dialogue_ID', 'Utterance_ID']).reset_index(drop=True)
if df_dev is not None:
    df_dev = df_dev.sort_values(['Dialogue_ID', 'Utterance_ID']).reset_index(drop=True)
if df_test is not None:
    df_test = df_test.sort_values(['Dialogue_ID', 'Utterance_ID']).reset_index(drop=True)

# Generate context_text
for df in [df_train, df_dev, df_test]:
    if df is not None:
        df['prev_clean_text'] = df.groupby('Dialogue_ID')['clean_text'].shift(1).fillna('')
        df['context_text'] = df.apply(
            lambda r: r['clean_text'] if r['prev_clean_text'] == '' 
            else r['prev_clean_text'] + " [SEP] " + r['clean_text'], 
            axis=1
        )

# Drop empty rows berdasarkan context_text
for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
    if df is not None:
        before = len(df)
        mask = df['context_text'].str.strip().str.len() > 0
        if name == 'train':
            df_train = df[mask].reset_index(drop=True)
        elif name == 'dev':
            df_dev = df[mask].reset_index(drop=True)
        else:
            df_test = df[mask].reset_index(drop=True)
        after = len(df[mask])
        print(f'{name}: {before} -> {after} (dropped {before - after} empty)')

print(f'\nContoh Context Text:')
for i in range(min(5, len(df_train))):
    print(f'  Speaker  : {df_train.iloc[i].get("Speaker", "Unknown")}')
    print(f'  Original : {df_train.iloc[i]["Utterance"]}')
    print(f'  Context  : {df_train.iloc[i]["context_text"]}\n')

In [ ]:
# === 4.3 TF-IDF Vectorization ===
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(df_train['context_text'])
X_dev_tfidf = tfidf.transform(df_dev['context_text'])
X_test_tfidf = tfidf.transform(df_test['context_text'])

# Encode labels
le = LabelEncoder()
le.fit(EMOTION_LABELS)
y_train = le.transform(df_train['Emotion'])
y_dev = le.transform(df_dev['Emotion'])
y_test = le.transform(df_test['Emotion'])

print(f'TF-IDF shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')
print(f'Labels: {le.classes_}')

## Section 5: Classical ML Models

In [ ]:
# === 5.1 Define Models & Hyperparameter Tuning ===
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline

print("Tuning LogisticRegression...")
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED))
])
lr_param_grid = {'clf__C': [0.1, 0.5, 1.0, 2.0, 5.0], 'clf__max_iter': [1000, 2000]}
lr_search = RandomizedSearchCV(lr_pipe, lr_param_grid, n_iter=5, cv=3, scoring='f1_weighted', n_jobs=-1, random_state=RANDOM_SEED)
lr_search.fit(df_train['context_text'], y_train)
print(f"Best LR params: {lr_search.best_params_}")

print("Tuning LinearSVC...")
svc_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ('clf', LinearSVC(class_weight='balanced', random_state=RANDOM_SEED))
])
svc_param_grid = {'clf__C': [0.1, 0.5, 1.0, 2.0, 5.0], 'clf__tol': [1e-4, 1e-3]}
svc_search = RandomizedSearchCV(svc_pipe, svc_param_grid, n_iter=5, cv=3, scoring='f1_weighted', random_state=RANDOM_SEED)
svc_search.fit(df_train['context_text'], y_train)
print(f"Best SVC params: {svc_search.best_params_}")

models = {
    'LogisticRegression (Tuned)': lr_search.best_estimator_,
    'LinearSVC (Tuned)': svc_search.best_estimator_,
    'MultinomialNB': Pipeline([('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', MultinomialNB(alpha=0.1))]),
    'RandomForest': Pipeline([('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', RandomForestClassifier(n_estimators=200, max_depth=50, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1))]),
    'GradientBoosting': Pipeline([('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_SEED))]),
}

# === 5.2 Train & Evaluate ===
results = {}
for name, model in tqdm(models.items(), desc='Training/Evaluating ML models'):
    model.fit(df_train['context_text'], y_train)
    y_pred = model.predict(df_dev['context_text'])
    acc = accuracy_score(y_dev, y_pred)
    f1_w = f1_score(y_dev, y_pred, average='weighted')
    f1_m = f1_score(y_dev, y_pred, average='macro')
    results[name] = {'accuracy': acc, 'f1_weighted': f1_w, 'f1_macro': f1_m, 'model': model}
    print(f'{name:28s} | Acc: {acc:.4f} | F1w: {f1_w:.4f} | F1m: {f1_m:.4f}')

# Best model
best_name = max(results, key=lambda k: results[k]['f1_weighted'])
print(f'\n🏆 Best ML model: {best_name} (F1w={results[best_name]["f1_weighted"]:.4f})')

In [ ]:
# === 5.3 Classification Report — Best ML Model ===
best_ml = results[best_name]['model']
y_pred_best = best_ml.predict(df_dev['context_text'])
print(f'Classification Report — {best_name} (Dev Set)\n')
print(classification_report(y_dev, y_pred_best, target_names=le.classes_))

In [ ]:
# === 5.4 Confusion Matrix — Best ML Model ===
cm = confusion_matrix(y_dev, y_pred_best)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## Section 6: Deep Learning — BiLSTM + Attention

In [ ]:
# === 6.1 Vocabulary & Dataset ===
from transformers import AutoTokenizer

import os
import glob
# Auto-detect model path in Kaggle
MODEL_PATH = "/kaggle/input/models/alexxxsem/deberta-v3/pytorch/base/2"
if not os.path.exists(MODEL_PATH):
    possible_configs = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if possible_configs:
        # Use the first valid huggingface model folder we find
        MODEL_PATH = os.path.dirname(possible_configs[0])


print("Loading tokenizer from:", MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

class TransformerDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

BATCH_SIZE = 32  # standard batch size for small transformers to fit on GPU easily
MAX_LEN = 64     # MELD utterances are short, 64 is more than enough and keeps it fast

print("Tokenizing train split...")
train_encodings = tokenizer(df_train['context_text'].tolist(), max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors="pt")
print("Tokenizing dev split...")
dev_encodings = tokenizer(df_dev['context_text'].tolist(), max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors="pt")
print("Tokenizing test split...")
test_encodings = tokenizer(df_test['context_text'].tolist(), max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors="pt")

# Strip the batch dimension from tensors so they can be processed by DataLoader
train_encodings = {key: val for key, val in train_encodings.items()}
dev_encodings = {key: val for key, val in dev_encodings.items()}
test_encodings = {key: val for key, val in test_encodings.items()}

train_ds = TransformerDataset(train_encodings, y_train)
dev_ds = TransformerDataset(dev_encodings, y_dev)
test_ds = TransformerDataset(test_encodings, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Max len: {MAX_LEN} | Batch: {BATCH_SIZE}')
print(f'Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}')

In [ ]:
# === 6.2 DeBERTa Model ===
from transformers import AutoModelForSequenceClassification

NUM_CLASSES = len(EMOTION_LABELS)
print("Loading model from:", MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    num_labels=NUM_CLASSES, 
    ignore_mismatched_sizes=True
)
model.to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

In [ ]:
# === 6.3 Training Loop ===
from torch.cuda.amp import autocast, GradScaler

# Class weights for imbalanced data
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
class_weights = torch.tensor(1.0 / (class_counts + 1e-6), dtype=torch.float32).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

criterion = nn.CrossEntropyLoss(weight=class_weights)
# Use a smaller learning rate suitable for fine-tuning transformers (2e-5)
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)
from transformers import get_linear_schedule_with_warmup
NUM_EPOCHS = 15
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

# Initialize GradScaler for mixed precision training
scaler = GradScaler()

best_f1 = 0
history = {'train_loss': [], 'dev_acc': [], 'dev_f1': []}

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} (Train)'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        optimizer.zero_grad()
        
        # Cast operations to mixed precision
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits.float(), labels)
            
        # Scale loss and backpropagate
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels']
            
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = outputs.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            
    dev_acc = accuracy_score(all_labels, all_preds)
    dev_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    history['train_loss'].append(avg_train_loss)
    history['dev_acc'].append(dev_acc)
    history['dev_f1'].append(dev_f1)
    
    
    print(f'Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Dev Acc: {dev_acc:.4f} | Dev F1: {dev_f1:.4f}')
    
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), 'best_deberta_model.pt')
        print(f'  🏆 New best model saved! (F1: {best_f1:.4f})')

In [ ]:
# === 6.4 Training Curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], 'b-o', markersize=3)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['dev_acc'], 'g-o', markersize=3, label='Accuracy')
axes[1].plot(history['dev_f1'], 'r-o', markersize=3, label='F1 Weighted')
axes[1].set_title('Dev Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

## Section 7: Model Comparison & Final Results

In [ ]:
# === 7.1 Load Best DeBERTa & Evaluate on Test Set ===
model.load_state_dict(torch.load('best_deberta_model.pt'))
model.eval()
deberta_preds, deberta_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels']
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        deberta_preds.extend(preds)
        deberta_labels.extend(labels.numpy())
deberta_test_acc = accuracy_score(deberta_labels, deberta_preds)
deberta_test_f1w = f1_score(deberta_labels, deberta_preds, average='weighted')
deberta_test_f1m = f1_score(deberta_labels, deberta_preds, average='macro')

# ML models on test set
comparison = []
for name, res in results.items():
    y_pred_test = res['model'].predict(df_test['context_text'])
    comparison.append({
        'Model': name,
        'Test Acc': accuracy_score(y_test, y_pred_test),
        'Test F1w': f1_score(y_test, y_pred_test, average='weighted'),
        'Test F1m': f1_score(y_test, y_pred_test, average='macro'),
    })
comparison.append({
    'Model': 'DeBERTa-v3-base',
    'Test Acc': deberta_test_acc,
    'Test F1w': deberta_test_f1w,
    'Test F1m': deberta_test_f1m,
})
df_comp = pd.DataFrame(comparison).sort_values('Test F1w', ascending=False).reset_index(drop=True)
print('╔══════════════════════════════════════════════════════════╗')
print('║            FINAL MODEL COMPARISON (Test Set)            ║')
print('╚══════════════════════════════════════════════════════════╝')
display(df_comp)

In [ ]:
# === 7.2 Visual Comparison ===
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df_comp))
w = 0.25
ax.bar([i - w for i in x], df_comp['Test Acc'], w, label='Accuracy', color='#3498DB')
ax.bar(x, df_comp['Test F1w'], w, label='F1 Weighted', color='#E74C3C')
ax.bar([i + w for i in x], df_comp['Test F1m'], w, label='F1 Macro', color='#2ECC71')
ax.set_xticks(x)
ax.set_xticklabels(df_comp['Model'], rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

best_overall = df_comp.iloc[0]
print(f'\n🏆 Best Overall: {best_overall["Model"]} '
      f'(F1w={best_overall["Test F1w"]:.4f}, Acc={best_overall["Test Acc"]:.4f})')

In [ ]:
# === 7.3 Classification Report & Confusion Matrix — Best Overall ===
if df_comp.iloc[0]['Model'] == 'DeBERTa-v3-base':
    final_preds, final_labels = deberta_preds, deberta_labels
else:
    best_ml_final = results[df_comp.iloc[0]['Model']]['model']
    final_preds = best_ml_final.predict(df_test['context_text'])
    final_labels = y_test

print(f'=== Classification Report — {df_comp.iloc[0]["Model"]} (Test Set) ===\n')
print(classification_report(final_labels, final_preds, target_names=le.classes_))

cm = confusion_matrix(final_labels, final_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {df_comp.iloc[0]["Model"]} (Test)')
plt.tight_layout()
plt.show()

## Section 8: Inference Demo

In [ ]:
# === 8.1 Inference Function ===
def predict_emotion(text, prev_text="", use_dl=True):
    """Prediksi emosi dari teks input dengan opsional konteks kalimat sebelumnya."""
    cleaned = clean_text(text)
    cleaned_prev = clean_text(prev_text) if prev_text else ""
    
    # Gabungkan dengan delimiter [SEP] jika ada konteks kalimat sebelumnya
    context = cleaned if not cleaned_prev else cleaned_prev + " [SEP] " + cleaned
    
    if use_dl:
        inputs = tokenizer(
            context,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        input_ids = inputs['input_ids'].to(DEVICE)
        attention_mask = inputs['attention_mask'].to(DEVICE)
        model.eval()
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred_idx = probs.argmax()
    else:
        pred_idx = best_ml.predict([context])[0]
        probs = None
    emotion = le.inverse_transform([pred_idx])[0]
    return emotion, probs

# === 8.2 Demo ===
test_pairs = [
    ("I am so happy to see you again!", ""),
    ("This is absolutely disgusting, I can't believe it.", ""),
    ("I'm really scared of what might happen next.", "Did you hear that sound?"),
    ("Whatever, I don't really care about it.", ""),
    ("How could you do this to me?! I'm furious!", "I just sold your computer."),
    ("I feel so lonely and sad without you.", "I have to leave for college tomorrow."),
    ("Oh my god! I can't believe you're here! What a surprise!", "Guess who just walked in?"),
]

print('🎯 Emotion Prediction Demo (DeBERTa-v3-base) with Dialogue Context\n')
print(f'{"Context / Text":<65} | {"Predicted":>10} | Confidence')
print('-' * 95)
for text, prev in test_pairs:
    emotion, probs = predict_emotion(text, prev, use_dl=True)
    conf = probs.max() if probs is not None else 0
    emoji = {'anger':'😠','disgust':'🤢','fear':'😨','joy':'😊',
             'neutral':'😐','sadness':'😢','surprise':'😲'}.get(emotion,'❓')
    
    display_str = text if not prev else f"[{prev}] -> {text}"
    # Truncate if too long for display formatting
    if len(display_str) > 62:
        display_str = display_str[:59] + "..."
        
    print(f'{display_str:<65} | {emoji} {emotion:>8} | {conf:.2%}')

In [ ]:
# === 8.3 Interactive — Coba sendiri! ===
# Ganti teks di bawah untuk test:
your_prev_text = "Did you see what they did?"
your_text = "I hate it when the goverment took our taxes!"
emotion, probs = predict_emotion(your_text, your_prev_text, use_dl=True)

print(f'Context: "{your_prev_text}"')
print(f'Input  : "{your_text}"')
print(f'Predicted Emotion: {emotion}')
if probs is not None:
    print('\nProbabilities:')
    for i, label in enumerate(le.classes_):
        bar = '█' * int(probs[i] * 40)
        print(f'  {label:10s} {probs[i]:.4f} {bar}')

## ✅ Notebook Selesai!**Summary:**

- Dataset MELD berhasil diproses (train/dev/test)
- 5 Classical ML models + 1 BiLSTM+Attention telah di-train
- Perbandingan model disajikan di Section 7- Inference demo tersedia di Section 8